# **ACE (Automatic Concept-based Explanations): Practice**

**Hi, friends.**

In the TCAV practice the concept “stripes” was given in advance: someone collected a `striped` folder of striped pictures, and TCAV tested whether that concept matters for the class “zebra”. ACE (Automatic Concept-based Explanations, [Ghorbani et al., 2019](https://arxiv.org/abs/1902.03129)) removes that step: the method finds candidate concepts on its own by cutting images of the class into pieces and grouping similar pieces by the network's activations. A person names a found concept afterwards, by looking at the pictures.

The authors published [TensorFlow code](https://github.com/amiratag/ACE). Here we use a simplified implementation of the same three steps in PyTorch and `captum`, on top of the data and model from the TCAV practice: the zebras from `tcav.zip`, GoogLeNet with ImageNet weights, the class “zebra” (index 340), and layer `inception4e`.

In this practice you will:
- cut zebra images into segments with the SLIC algorithm at two resolutions and feed each segment to the network on a gray background;
- turn the segments into activations of layer `inception4e` and group them with KMeans — each cluster becomes a candidate concept;
- drop clusters assembled from too few images and look at the segments of the remaining ones;
- compute the TCAV score of every remaining cluster against random images and order the concepts by importance;
- check how much the list of concepts depends on the number of clusters $k$.

**Happy coding!**

As always, we first install the dependencies. Colab already has `scikit-image`, `scikit-learn` and `torch`; only `captum` has to be installed.

In [ ]:
!pip install captum==0.9.0 -q

In [ ]:
import os, glob, random, shutil, warnings

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torchvision
from torchvision import transforms

from skimage.segmentation import slic, mark_boundaries
from sklearn.cluster import KMeans

from captum.concept import TCAV, Concept

warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

## Data and model

The data are the same as in the TCAV practice: the folder `tcav/data/zebra` with zebras and the folders `tcav/concepts/random500_0`, `tcav/concepts/random500_1` with random photos from Wikimedia Commons (Unsplash photos, CC0). The source and license of every photo are in the file `tcav/CREDITS.md`. We will not need the folders `striped`, `zigzagged`, `dotted`: we are going to find the concepts ourselves.

In [ ]:
!wget -q https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/tcav.zip -O tcav.zip

!unzip -q -o 'tcav.zip'

To keep the run on a Colab CPU within a few minutes, we take the first 20 zebras. Every image is brought to 224×224 the same way as in TCAV (`Resize(256)` and `CenterCrop(224)`), but without normalization for now: we are going to segment the plain picture.

In [ ]:
general_path = os.path.abspath('tcav')  # the tcav folder in the working directory: /content/tcav on Colab

crop = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])
normalize = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

def load_images(folder, n=None):
    files = sorted(glob.glob(os.path.join(general_path, folder, '*.jpg')))[:n]  # fixed file order
    return [np.array(crop(Image.open(f).convert('RGB'))) for f in files]      # uint8, (224, 224, 3)

N_IMAGES = 20
zebra_imgs = load_images('data/zebra', N_IMAGES)
print(f'Zebras: {len(zebra_imgs)}, image size: {zebra_imgs[0].shape}')

In [ ]:
model = torchvision.models.googlenet(weights=torchvision.models.GoogLeNet_Weights.IMAGENET1K_V1).to(device).eval()

zebra_ind = 340
zebra_tensors = torch.stack([normalize(img) for img in zebra_imgs]).to(device)
with torch.no_grad():
    zebra_pred = model(zebra_tensors).argmax(1)
print(f'The model assigns {(zebra_pred == zebra_ind).sum().item()} of {N_IMAGES} images to the class "zebra"')

## Step 1. Cutting into segments

ACE cuts every image with the [SLIC](https://scikit-image.org/docs/stable/api/skimage.segmentation.html#skimage.segmentation.slic) algorithm: it groups neighboring pixels of similar color into superpixels, and the `n_segments` parameter sets roughly how many pieces come out. The paper uses three resolutions so that a concept can be a large part of the object as well as a small texture; the authors' code uses `n_segments` = 15, 50 and 80 with `compactness=20`. We take two of them, 15 and 50: on 20 zebras resolution 80 yields more than half of all segments and roughly doubles the run time.

Let us see how one image is cut at these two resolutions.

In [ ]:
RESOLUTIONS = [15, 50]

def segment(img, n_segments):
    return slic(img, n_segments=n_segments, compactness=20, sigma=1, start_label=0)

fig, axs = plt.subplots(1, len(RESOLUTIONS), figsize=(10, 5))
for ax, n_seg in zip(axs, RESOLUTIONS):
    labels = segment(zebra_imgs[7], n_seg)
    ax.imshow(mark_boundaries(zebra_imgs[7], labels))
    ax.set_title(f'n_segments={n_seg}: {labels.max() + 1} segments')
    ax.axis('off')
plt.show()

Next, every segment is fed to the network separately. ACE does it like this: it fills the pixels outside the segment with gray of value 117 in every channel, cuts out the rectangle around the segment (the bounding box), and stretches it to the network's input size of 224×224.

**Task 1.** Complete the function `segment_patch`: it takes an image `img` (uint8, 224×224×3) and a boolean segment mask `mask` (224×224), and returns a uint8 array of 224×224×3: the segment on a gray background, cut by its bounding box and stretched to 224×224 (`Image.fromarray(...).resize((224, 224), Image.BICUBIC)`).

In [ ]:
GRAY = 117

def segment_patch(img, mask):
    # Your code here
    pass

In [ ]:
EXAMPLE = 7
labels = segment(zebra_imgs[EXAMPLE], 15)
example_masks = [labels == labels[y, x] for y, x in [(60, 100), (130, 200), (40, 30), (215, 130)]]

fig, axs = plt.subplots(2, 4, figsize=(16, 8))
for j, mask in enumerate(example_masks):
    axs[0, j].imshow(mark_boundaries(zebra_imgs[EXAMPLE], mask.astype(int)))
    axs[0, j].set_title(f'segment {j}: {mask.mean():.1%} of the image')
    axs[1, j].imshow(segment_patch(zebra_imgs[EXAMPLE], mask))
    axs[0, j].axis('off'); axs[1, j].axis('off')
plt.show()

**Self-check question.** Why is the background gray with value 117 and not black? Compute what 117 and 0 become after `normalize`.

<details><summary>Check yourself</summary>

After ImageNet normalization the pixel value 117 maps to values near zero in all three channels: −0.11, 0.01 and 0.23 (computed in the code below), that is, almost to the “average” input the network was trained on. A black pixel 0 maps to −2.12, −2.04 and −1.80: the network would see a strong dark-frame signal around the segment that is not in the original image.

</details>

In [ ]:
mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
print('117 ->', np.round((117 / 255 - mean) / std, 2))
print('  0 ->', np.round((0 - mean) / std, 2))

Before passing the segments to the network, ACE drops two kinds of extra ones. The first are very small segments, under 0.1% of the image area: after stretching to 224×224 only a few blurry pixels remain. The second are repeats: a large piece at resolution 15 and a piece at resolution 50 can almost coincide, and then the same part of the image would enter the data twice. The ACE code treats a segment as a repeat if its Jaccard index with some segment of the same image found at an earlier resolution is above 0.5. The Jaccard index of two masks is the number of pixels in both masks divided by the number of pixels in at least one of them.

**Task 2.** Complete the function `is_duplicate`: it takes a boolean mask `mask` and a list `seen_masks` of masks of the same image from earlier resolutions, and returns `True` if the Jaccard index of `mask` with at least one mask from `seen_masks` is above 0.5. Then run the code below: it cuts all 20 zebras at the two resolutions and keeps the segments that pass both filters.

In [ ]:
MIN_AREA = 0.001

def is_duplicate(mask, seen_masks):
    # Your code here
    pass

In [ ]:
patches, owners, scales = [], [], []   # segment on gray background, image index, SLIC resolution
n_raw, n_small, n_dup = 0, 0, 0
for i, img in enumerate(zebra_imgs):
    seen_masks = []                      # masks of this image from earlier resolutions
    for n_seg in RESOLUTIONS:
        labels = segment(img, n_seg)
        new_masks = []
        for s in np.unique(labels):
            mask = labels == s
            n_raw += 1
            if mask.mean() <= MIN_AREA:
                n_small += 1
            elif is_duplicate(mask, seen_masks):
                n_dup += 1
            else:
                new_masks.append(mask)
                patches.append(segment_patch(img, mask))
                owners.append(i)
                scales.append(n_seg)
        seen_masks += new_masks
owners, scales = np.array(owners), np.array(scales)

print(f'Segments in total: {n_raw}; small: {n_small}; repeats: {n_dup}; kept: {len(patches)}')
for n_seg in RESOLUTIONS:
    print(f'  n_segments={n_seg}: {(scales == n_seg).sum()} segments kept')

**Self-check question.** How many segments did SLIC produce on all 20 images, how many of them were dropped as repeats, and how many remained? At which resolution were the repeats dropped, and why not at the other one?

<details><summary>Check yourself</summary>

SLIC produced 982 segments: 238 at resolution 15 and 744 at resolution 50. There are no small segments under 0.1% of the image area; 28 segments were dropped as repeats, and 954 remained. All 28 repeats were dropped at resolution 50: a segment is compared only with masks of the same image from earlier resolutions, and at the first resolution, 15, there is nothing to compare with; segments of one resolution do not overlap each other.

</details>

## Step 2. Activations and clusters

We pass every segment through GoogLeNet and take the activation of layer `inception4e` with a hook. The activation map has shape 832×14×14; we average it over space and get a vector of 832 numbers per segment. We also store which class the model predicts for the segment itself: it will be useful at the end.

In [ ]:
LAYER = 'inception4e'
store = {}
hook = getattr(model, LAYER).register_forward_hook(lambda m, inp, out: store.update(act=out))

feats, seg_pred = [], []
with torch.no_grad():
    for b in range(0, len(patches), 64):
        x = torch.stack([normalize(p) for p in patches[b:b + 64]]).to(device)
        seg_pred.append(model(x).argmax(1))
        feats.append(store['act'].mean(dim=(2, 3)))   # average over the 14×14 space
hook.remove()

feats = torch.cat(feats).cpu().numpy()
seg_pred = torch.cat(seg_pred).cpu().numpy()
feats.shape

Now we group the segments from all images with KMeans. Every cluster is a candidate concept: a set of pieces that layer `inception4e` considers similar. The paper and the authors' code use $k=25$; we have 20 images and two SLIC resolutions, so fewer segments, and we take $k=12$.

In [ ]:
K = 12
kmeans = KMeans(n_clusters=K, n_init=10, random_state=0).fit(feats)
cluster_of = kmeans.labels_              # cluster index of every segment
dist = kmeans.transform(feats)           # distance of every segment to every cluster center
np.bincount(cluster_of)

## Step 3. Cleaning the clusters

Not every cluster is fit to be a concept. If all segments of a cluster come from one or two images, it is more likely a feature of those particular photos (the same background, for example) than something the network sees in the class “zebra” in general. ACE keeps clusters that have enough segments taken from many different images.

**Task 3.** Complete the function `select_clusters`: it returns the list of cluster indices that have at least `min_size` segments, taken from at least `min_images` different images (the image index of every segment is in `owners`). We use `min_size=20` and `min_images=10`, that is, half of the 20 zebras. This is a simplified version of the ACE filter: the authors' code has three conditions, which compare the number of images in a cluster both with the cluster size and with the number of all images of the class.

In [ ]:
MIN_SIZE, MIN_IMAGES = 20, 10

def select_clusters(cluster_of, owners, min_size=MIN_SIZE, min_images=MIN_IMAGES):
    # Your code here
    pass

In [ ]:
def cluster_table(cluster_of, kept):
    print('cluster | segments | images | n_segments 15/50 | predicted "zebra" | kept')
    for c in range(cluster_of.max() + 1):
        m = cluster_of == c
        per_scale = '/'.join(str((scales[m] == r).sum()) for r in RESOLUTIONS)
        print(f'{c:7d} | {m.sum():8d} | {len(np.unique(owners[m])):6d} | {per_scale:>15} | '
              f'{(seg_pred[m] == zebra_ind).mean():17.0%} | {"yes" if c in kept else "no"}')

kept = select_clusters(cluster_of, owners)
cluster_table(cluster_of, kept)
print(f'\nKept clusters: {len(kept)} of {K}: {kept}')

**Self-check question.** How many of the 12 clusters passed the filter? Which of the dropped clusters is assembled from the fewest images?

<details><summary>Check yourself</summary>

10 of the 12 clusters passed the filter. Dropped are cluster 1 (64 segments from 8 images) and cluster 9 (58 segments from 7 images); cluster 9 has the fewest images: its segments come from only seven photos, and such a cluster describes these photos rather than the class “zebra”.

</details>

Let us look at the candidate concepts themselves. For every remaining cluster we show the 6 segments closest to its center: ACE does the same when it shows the concepts it found.

In [ ]:
def show_clusters(clusters, cluster_of, dist, titles, n=6):
    fig, axs = plt.subplots(len(clusters), n, figsize=(2 * n, 2.1 * len(clusters)))
    for row, c in enumerate(clusters):
        members = np.where(cluster_of == c)[0]
        nearest = members[np.argsort(dist[members, c])[:n]]
        for j in range(n):
            axs[row, j].axis('off')
            if j < len(nearest):
                axs[row, j].imshow(patches[nearest[j]])
        axs[row, 0].set_title(titles[row], loc='left', fontsize=11)
    plt.tight_layout()
    plt.show()

show_clusters(kept, cluster_of, dist, [f'cluster {c}: {(cluster_of == c).sum()} segments' for c in kept])

## Step 4. TCAV score of every cluster

Now every remaining cluster is a concept in the TCAV sense: a set of example pictures. The rest is as in the TCAV practice: for a pair “cluster against random images” `captum` trains a linear classifier on the `inception4e` activations, takes its normal as the concept vector (CAV), and counts the fraction of zebras for which the derivative of the class 340 logit along the CAV is positive. That is the TCAV score (`sign_count`).

From every cluster we take the 40 segments closest to its center, so that the number of concept examples is close to the number of random images (38 and 41 in `random500_0` and `random500_1`). Every cluster is compared with two random sets, and the two TCAV scores are averaged. The paper uses dozens of random sets and calls a concept important when its TCAV score differs statistically from that of a random concept; with two sets there is no such test, so a TCAV score near 0.5 cannot be told apart from noise here.

In [ ]:
N_CONCEPT = 40

def make_concept(cid, name, imgs):
    tensors = [normalize(img).to(device) for img in imgs]
    return Concept(id=cid, name=name, data_iter=torch.utils.data.DataLoader(tensors, batch_size=64))

random_0_imgs, random_1_imgs = load_images('concepts/random500_0'), load_images('concepts/random500_1')
print(f'random500_0: {len(random_0_imgs)} images, random500_1: {len(random_1_imgs)} images')
random_0_concept = make_concept(100, 'random500_0', random_0_imgs)
random_1_concept = make_concept(101, 'random500_1', random_1_imgs)

def cluster_concept(c, cluster_of, dist):
    members = np.where(cluster_of == c)[0]
    nearest = members[np.argsort(dist[members, c])[:N_CONCEPT]]
    return make_concept(c, f'cluster_{c}', [patches[j] for j in nearest])

def cluster_tcav(clusters, cluster_of, dist, save_path):
    shutil.rmtree(save_path, ignore_errors=True)   # captum caches CAVs on disk by concept id
    tcav = TCAV(model=model, layers=[LAYER], save_path=save_path)   # sensitivity is the gradient with respect to the layer activations, the Captum default
    concepts = [cluster_concept(c, cluster_of, dist) for c in clusters]
    experimental_sets = [[cc, r] for cc in concepts for r in (random_0_concept, random_1_concept)]

    # Seeds before training the CAV classifiers: captum shuffles examples with random and trains the classifier in torch
    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)
    scores = tcav.interpret(inputs=zebra_tensors, experimental_sets=experimental_sets, target=zebra_ind)
    return {c: np.mean([scores[f'{c}-{r}'][LAYER]['sign_count'][0].item() for r in (100, 101)]) for c in clusters}

In [ ]:
tcav_scores = cluster_tcav(kept, cluster_of, dist, './cav_k12/')
ranking = sorted(kept, key=lambda c: -tcav_scores[c])
for place, c in enumerate(ranking, 1):
    print(f'{place:2d}. cluster {c:2d}: TCAV score = {tcav_scores[c]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar([f'cluster {c}' for c in ranking], [tcav_scores[c] for c in ranking], color='mediumspringgreen')
ax.bar_label(bars, fmt='%.3f')
ax.axhline(0.5, color='gray', ls='--', lw=1)
ax.set_ylim(0, 1.1)
ax.set_ylabel('TCAV score (sign_count), mean of 2 random sets')
ax.set_title(f'ACE concepts for class "zebra", layer {LAYER}, k={K}')
plt.xticks(rotation=45)
plt.show()

The bars are ordered by TCAV score from highest to lowest; the dashed line is 0.5, the level a concept with no influence on the class would give. Let us look at the segments of the clusters in the same order.

In [ ]:
show_clusters(ranking, cluster_of, dist, [f'#{p}: cluster {c}, TCAV score {tcav_scores[c]:.3f}' for p, c in enumerate(ranking, 1)])

**Task 4 (self-check question).** Which cluster got the highest TCAV score, what is its value, and what do the segments of this cluster look like? What name would you give this concept?

<details><summary>Check yourself</summary>

The highest TCAV score, 0.375, goes to cluster 6: its segments are close-ups of the stripes on the zebra's head and neck, and a fitting name for it is “zebra stripes”. Next come cluster 10 (0.225) with thin stripes on the legs and along the edges of the body and cluster 5 (0.175) with large striped pieces of the body. Cluster 5 is the only cluster where more than half of the segments come from resolution 15 (59 of 78).

No cluster reached 0.5: at layer `inception4e` the derivative of the “zebra” logit along the direction of every concept found is positive for fewer than half of the zebras. The concept `striped` from the TCAV practice behaves the same way at this layer: 0.34 and 0.17 against the two random sets. With two random sets such values cannot be told apart from noise.

</details>


## Limitation: a cluster is not yet a concept

We named the concept by looking at 6 pictures, while the cluster itself is a KMeans result with a $k$ that we chose. Let us check how much the found concepts depend on that choice: we repeat steps 2–4 on the same activations with $k=6$ and compare.

For every concept of the $k=12$ run we find the most similar cluster of the $k=6$ run by the Jaccard index: the number of segments in both clusters divided by the number of segments in at least one of them. An index of 1 means the cluster was reproduced in full; a value near 0 means there is no such concept at $k=6$.

In [ ]:
K_ALT = 6
kmeans_alt = KMeans(n_clusters=K_ALT, n_init=10, random_state=0).fit(feats)
cluster_alt, dist_alt = kmeans_alt.labels_, kmeans_alt.transform(feats)
kept_alt = select_clusters(cluster_alt, owners)
tcav_alt = cluster_tcav(kept_alt, cluster_alt, dist_alt, './cav_k6/')
ranking_alt = sorted(kept_alt, key=lambda c: -tcav_alt[c])

print(f'k={K_ALT}: kept {len(kept_alt)} of {K_ALT} clusters')
for place, c in enumerate(ranking_alt, 1):
    print(f'{place:2d}. cluster {c:2d}: {(cluster_alt == c).sum():4d} segments, TCAV score = {tcav_alt[c]:.3f}')

print(f'\nk={K} concept -> most similar k={K_ALT} cluster')
for place, c in enumerate(ranking, 1):
    a = cluster_of == c
    jac = [(a & (cluster_alt == d)).sum() / (a | (cluster_alt == d)).sum() for d in range(K_ALT)]
    d = int(np.argmax(jac))
    alt = f'TCAV score {tcav_alt[d]:.3f}' if d in kept_alt else 'dropped by the filter'
    print(f'#{place:2d} cluster {c:2d} (TCAV score {tcav_scores[c]:.3f}) -> cluster {d} (Jaccard {jac[d]:.2f}, {alt})')

In [ ]:
show_clusters(ranking_alt, cluster_alt, dist_alt, [f'k={K_ALT}, #{p}: cluster {c}, TCAV score {tcav_alt[c]:.3f}' for p, c in enumerate(ranking_alt, 1)])

**What the $k=6$ experiment showed.** At $k=6$, 5 of the 6 clusters passed the filter, and none of them reached a TCAV score of 0.5: cluster 4 (237 segments) got 0.275, cluster 1 (164 segments) 0.175, cluster 3 (165 segments) 0.100, cluster 2 (124 segments) 0.050, cluster 0 (191 segments) 0.025. The top concepts found at $k=12$ were redistributed among them: clusters 10 and 5 (thin stripes and large striped pieces) merged into cluster 1 with Jaccard indices 0.32 and 0.48; cluster 6 (stripes on the head and neck) is most similar to cluster 2, but with an index of only 0.18; cluster 2 (striped pieces mixed with unrelated ones) went into cluster 4 together with cluster 0 (dark pieces with a light stripe). None of the concepts found at $k=12$ was reproduced at $k=6$ with a Jaccard index above 0.49.

How many stripe concepts are found, where the lines between them run, and what TCAV score each one gets is set by the $k$ we chose: cluster 6, which has the highest TCAV score of 0.375 at $k=12$, is most similar to a cluster with a TCAV score of 0.050 at $k=6$. A cluster is a KMeans result, and it is the person looking at it who turns it into a concept.


## Conclusion

In this practice you went through the three ACE steps on 20 zebras:
- SLIC at resolutions 15 and 50 produced 982 segments, and 954 remained after dropping repeats; every segment was fed to the network on a gray 117 background;
- KMeans on the averaged activations of layer `inception4e` split them into 12 clusters, and the filter on the number of segments and images kept 10;
- no cluster reached a TCAV score of 0.5 against the two random sets. Three clusters with zebra stripes rank above the rest: stripes on the head and neck (cluster 6, 0.375), thin stripes (cluster 10, 0.225) and large pieces of the body (cluster 5, 0.175); dark pieces with a light stripe (cluster 0) and striped pieces mixed with unrelated ones (cluster 2) got 0.125 and 0.100, and the other five clusters (green and dry grass, blurred background, plain light patches) got at most 0.025.

All three limitations from the theory are visible on these data:
- **a cluster is not yet a concept:** at $k=6$ the striped concepts regrouped, cluster 6 is most similar to a cluster with a TCAV score of 0.050, and we named every cluster ourselves by looking at 6 pictures;
- **the segmentation decides for the method:** the large striped pieces of the body (cluster 5) are assembled mostly from segments of resolution 15 (59 of 78); without the coarse resolution this concept would not exist;
- **scraps on a gray background are out of distribution:** the model itself assigns the segments of cluster 6, which has the highest TCAV score of 0.375, to the class “zebra” only 9% of the time, although it assigns the whole images to it in 20 cases of 20; only cluster 5 with large pieces of the body has a high share (73%), and its TCAV score is 0.175. TCAV measures how sensitive the logit is to a direction in activation space, not whether the model recognizes a zebra in the scrap.

One more simplification: the paper compares a concept's TCAV score with those of random concepts using a statistical test over dozens of random sets, while we have two sets, so values such as 0.375 and 0.225 for clusters 6 and 10 cannot be told apart from noise.

The original TensorFlow implementation of ACE is at [github.com/amiratag/ACE](https://github.com/amiratag/ACE).

**Thank you for your work and attention!**